# 02 · msci-code: one coder for completion and chat

One **Qwen2.5-Coder-1.5B** student does both jobs in the IDE: inline completion (fill-in-the-middle) and
chat (write, explain, document and fix Python). The teacher is **Qwen2.5-Coder-14B-Instruct-AWQ** on vLLM.
`msci-code-fast` stays live for autocomplete until this model beats it on the same FIM spans.

| SFT mix | Source | Target | Check |
|---|---|---|---|
| FIM (~2,500) | spans from the pinned repos, same as msci-code-fast | the original code | ground truth |
| MBPP (~380) | MBPP train problems | teacher, 4 answers | MBPP's own tests |
| synth (~2,000 seeds) | problems the teacher writes from real repo functions | teacher, 4 answers | reference passes its tests **and** ≥ 2 answers agree |
| docstring (~600) | repo functions with docstrings removed | the original function | ground truth |
| explain (~400) | repo functions | teacher | length rules |

**Runtime: L4** (Runtime → Change runtime type → L4 GPU, High RAM on). Time estimates below are guesses
until sections 2–3 measure them:

| Step | What happens | L4 time (est.) |
|---|---|---|
| prepare | clone repos, FIM spans (identical to msci-code-fast), MBPP, repo functions | ~10 min |
| teacher_generate | vLLM: write problems, 4 answers per problem, explanations | **section 3 projects it** (2–5 h guess) |
| verify | run every answer against its tests in a sandbox | ~10–20 min |
| teacher_logits / train_kd | skipped (`train.kd.enabled: false`; distillation hurt msci-code-fast) | – |
| train_sft | bf16 LoRA r=32, 2 epochs | ~1 h |
| evaluate | FIM (vs msci-code-fast) + chat pass@1 for base / SFT; gates | ~20 min |
| quantise | merge, GGUF q8_0, CPU latency of the GGUF | ~10 min |
| register | model store, model card, catalogue.yaml | <1 min |

Every step resumes after a disconnect: re-run the setup cell, the variables cell and the same step cell.

In [ ]:
# --- Setup: GPU check, Google Drive, repo, packages (run every session) ---
import glob, json, os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
SVLM_ROOT = '/content/drive/MyDrive/svlm'       # run outputs, checkpoints and the model store live here
REPO_ON_DRIVE = f'{SVLM_ROOT}/svlm-catalogue'   # copy the repo folder here once (Drive for Desktop works)
REPO = '/content/svlm-catalogue'                # working copy on Colab's fast local disk
os.environ['SVLM_ROOT'] = SVLM_ROOT
os.makedirs(SVLM_ROOT, exist_ok=True)

def _find_repo():
    """Return the first folder that looks like the repo (has configs/ and svlm/), searching
    Colab's local disk and Drive, including one extra nesting level from zip/folder uploads."""
    # Drive first: re-running this cell then refreshes the local copy with whatever was synced to Drive
    bases = [REPO_ON_DRIVE, '/content/drive/MyDrive/svlm-catalogue', '/content/svlm-catalogue', '/content/svlm', '/content']
    for b in bases:
        for cand in (b, os.path.join(b, 'svlm-catalogue')):
            if os.path.isdir(os.path.join(cand, 'configs')) and os.path.isdir(os.path.join(cand, 'svlm')):
                return cand
    return None

found = _find_repo()
if found is None:
    from google.colab import files
    print('Repo not found. Upload svlm-catalogue.zip (from C:\\projects\\TechFest):')
    up = files.upload()
    subprocess.run(['unzip', '-q', '-o', next(iter(up)), '-d', '/content'], check=True)
    found = _find_repo()
    assert found, 'zip did not contain the svlm-catalogue folder'
if found.startswith('/content/drive'):
    # work from a copy on Colab's local disk: faster, and Drive stays the clean source
    subprocess.run(['rm', '-rf', REPO], check=True)
    subprocess.run(['cp', '-r', found, REPO], check=True)
    found = REPO
REPO = found
print('repo:', REPO)
os.chdir(REPO)
sys.path.insert(0, REPO)

locks = sorted(glob.glob(f'{SVLM_ROOT}/env/lock-colab-*.txt'))
REQ = locks[-1] if locks else 'requirements/colab.txt'
print('installing from', REQ)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQ], check=True)
# Colab preinstalls an old torchao; recent peft refuses to build LoRA layers while it is present (unused here)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=False)

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} | compute {p.major}.{p.minor} | {p.total_memory/1e9:.1f} GB | bf16: {torch.cuda.is_bf16_supported(including_emulation=False)}')

## 0 · vLLM

vLLM brings its own torch build. Install it once per runtime, then check that training libraries still import.
If the install fails or breaks the check below, use the slow fallback instead: in section 4 add
`--set teacher.backend=hf` (4-bit transformers; expect several times longer teacher steps).

In [ ]:
!pip install -q -r requirements/vllm.txt
!python -c "import vllm, torch, transformers, peft; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)"

## 1 · Unit tests (CPU: chat format, sandbox, problem parsing, verify, gates)

In [ ]:
!python -m pytest -q tests/test_core.py tests/test_code_task.py

## 2 · Pick the starting checkpoint

Scores the **untrained** Base and Instruct 1.5B coders on msci-code-fast's held-out FIM spans and on MBPP
validation (tests run in the sandbox). Training restores FIM easily (2,500 original spans in the mix); chat
ability is harder to create from nothing, so prefer the better chat score unless its FIM score is far behind.
~10 min.

In [ ]:
!python tools/compare_bases.py --config configs/code.yaml --n-fim 150

In [ ]:
# Set the starting checkpoint from the table above, then run this cell (and re-run it after any reconnect).
STUDENT = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
VERSION = '0.1.0'
SETS = f'--set student.model={STUDENT} --set version={VERSION}'
from pathlib import Path
from svlm.utils import read_json, read_jsonl
RUN = Path(SVLM_ROOT) / 'runs' / 'msci-code' / f'{VERSION}-l4'
print(SETS)

## 3 · Teacher throughput

Loads the 14B AWQ teacher in vLLM in its own process, answers 48 MBPP problems 4 times, and projects the
full teacher step for this config. Use the projection to plan the session (Colab Pro sessions last up to 24 h,
and every shard is saved, so a disconnect costs at most one shard).

In [ ]:
!python tools/bench_chat_teacher.py --config configs/code.yaml --problems 48

## 4 · Smoke run (tiny data, real models, ~30–40 min)

All steps with a few dozen examples per type, as `0.0.0-smoke`, so nothing touches the real run. Gates
will fail at this size (and `fim_vs_reference` cannot compare: the smoke held-out spans differ);
`register.force=true` registers it as non-live so the model card can be checked.

In [ ]:
SMOKE = ('--set version=0.0.0 --set tier=smoke --set data.fim.n_train=200 --set data.fim.n_heldout=40 '
         '--set data.code.fim_train=150 --set data.code.mbpp_max=40 --set data.code.synth_train=60 '
         '--set data.code.synth_heldout=20 --set data.code.docstring=30 --set data.code.explain=20 '
         '--set train.sft.epochs=1 --set register.force=true')
!python pipelines/run_local.py --config configs/code.yaml --set student.model={STUDENT} {SMOKE}

In [ ]:
smoke = Path(SVLM_ROOT) / 'model_store' / 'msci-code' / '0.0.0-smoke' / 'model_card.md'
from IPython.display import Markdown, display
display(Markdown(smoke.read_text()))

## 5 · Full run

One cell per step. `teacher_generate` runs vLLM in a child process, so GPU memory is free again for training
in the same session.

In [ ]:
!python pipelines/run_local.py --config configs/code.yaml {SETS} --status

### prepare

In [ ]:
!python pipelines/run_local.py --config configs/code.yaml {SETS} --steps prepare

In [ ]:
m = read_json(RUN / 'prepare' / 'manifest.json')
print(json.dumps({k: m[k] for k in ('fim', 'chat_train', 'chat_heldout', 'decontam_dropped')}, indent=2))
ex = [r for r in read_jsonl(RUN / 'prepare' / 'chat_train.jsonl') if r['type'] == 'synth_seed'][0]
print('--- a repository function that seeds a teacher-written problem:', ex['path']); print(ex['code'])

### teacher_generate
The long step. Shards of 256 are saved to Drive as they finish.

In [ ]:
!python pipelines/run_local.py --config configs/code.yaml {SETS} --steps teacher_generate

### verify
Every teacher answer is run against its tests in a sandbox.

In [ ]:
!python pipelines/run_local.py --config configs/code.yaml {SETS} --steps verify

In [ ]:
r = read_json(RUN / 'verify' / 'filter_report.json')
print(json.dumps({k: r[k] for k in ('teacher_train_filter', 'sft_mix', 'chat_eval', 'teacher_heldout')}, indent=2))
row = next(x for x in read_jsonl(RUN / 'verify' / 'sft.jsonl') if x.get('type') == 'synth')
print('--- user ---'); print(row['messages'][0]['content']); print('--- assistant (kept: passed the tests) ---'); print(row['messages'][1]['content'])

### teacher_logits
Skipped unless `train.kd.enabled=true`.

In [ ]:
!python pipelines/run_local.py --config configs/code.yaml {SETS} --steps teacher_logits

### train_sft

In [ ]:
!python pipelines/run_local.py --config configs/code.yaml {SETS} --steps train_sft

In [ ]:
import matplotlib.pyplot as plt
h = read_json(RUN / 'train_sft' / 'train_log.json')['log_history']
pts = [(r['step'], r['loss']) for r in h if 'loss' in r]
plt.plot(*zip(*pts)); plt.xlabel('step'); plt.ylabel('loss'); plt.title('train_sft'); plt.grid(alpha=.3); plt.show()

### train_kd
Skipped unless `train.kd.enabled=true`.

In [ ]:
!python pipelines/run_local.py --config configs/code.yaml {SETS} --steps train_kd

### evaluate

In [ ]:
!python pipelines/run_local.py --config configs/code.yaml {SETS} --steps evaluate

In [ ]:
m = read_json(RUN / 'evaluate' / 'metrics.json')
ref = m.get('fim_reference', {})
print('FIM (same held-out spans as msci-code-fast)')
if ref.get('status') == 'ok': print(f"  reference msci-code-fast:{ref['version']}  edit-sim {ref['edit_similarity']:.3f}")
else: print('  reference:', ref.get('status'))
for k in ('base', 'sft', 'final'):
    f = m[k]['fim']; print(f"  {k:6s} exact {f['exact_match']:.3f}  edit-sim {f['edit_similarity']:.3f}  parse {f['parse_rate']:.3f}")
print('Chat pass@1 (held-out problems, tests in sandbox)')
print(f"  teacher {m['teacher']['chat']['pass@1']}  {m['teacher']['chat']['by_type']}")
for k in ('base', 'sft', 'final'): print(f"  {k:7s} {m[k]['chat']['pass@1']}  {m[k]['chat']['by_type']}")
print('\nGATES:', m['gates']['verdict'].upper())
for k, v in m['gates']['checks'].items(): print(f"  {'PASS' if v['pass'] else 'FAIL'}  {k}: {v['value']} (threshold {v.get('threshold', v.get('base'))})")

### quantise
Also measures time to first token of the GGUF on this machine's CPU (indicative: Colab's CPU is not your server).

In [ ]:
!python pipelines/run_local.py --config configs/code.yaml {SETS} --steps quantise

In [ ]:
print(json.dumps(read_json(RUN / 'quantise' / 'summary.json').get('cpu_latency'), indent=2))

### register
Fails on purpose if a gate failed; add `--set register.force=true` to store it as non-live.

In [ ]:
!python pipelines/run_local.py --config configs/code.yaml {SETS} --steps register

In [ ]:
from IPython.display import Markdown, display
display(Markdown((Path(SVLM_ROOT) / 'model_store' / 'msci-code' / f'{VERSION}-l4' / 'model_card.md').read_text()))
print(open(Path(SVLM_ROOT) / 'model_store' / 'catalogue.yaml').read())

## 6 · Try it: chat and inline completion with the merged model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from svlm.chat import format_chat, trim_reply
from svlm.fim import fim_prompt, trim_completion
mdir = RUN / 'merged'
tok = AutoTokenizer.from_pretrained(mdir); model = AutoModelForCausalLM.from_pretrained(mdir, dtype=torch.bfloat16).cuda()

def ask(question, max_new_tokens=400):
    ids = tok(format_chat([{'role': 'user', 'content': question}]), return_tensors='pt', add_special_tokens=False).to('cuda')
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return trim_reply(tok.decode(out[0][ids['input_ids'].shape[1]:]))

print(ask('Write a function that returns the maximum drawdown of a pandas Series of prices.'))
print('\n' + '=' * 60 + '\n')
prefix = "import numpy as np\n\ndef sharpe_ratio(returns, rf=0.0, periods=252):\n"
suffix = "\n    return mean / std * np.sqrt(periods)\n"
ids = tok(fim_prompt(prefix, suffix), return_tensors='pt', add_special_tokens=False).to('cuda')
out = model.generate(**ids, max_new_tokens=64, do_sample=False)
print(prefix + '\033[92m' + trim_completion(tok.decode(out[0][ids['input_ids'].shape[1]:]), 'block', suffix) + '\033[0m' + suffix)

## 7 · Serve it from your machine

1. Copy `MyDrive/svlm/model_store/` (the new `msci-code/` folder and `catalogue.yaml`) into
   `C:\projects\TechFest\model_store\`.
2. Restart the gateway. It loads every live model in `catalogue.yaml`: `msci-code-fast` and `msci-code`.
3. In `~/.continue/config.yaml`, add the `msci-code` entry from `gateway/continue-config.yaml` (chat, edit,
   apply). Move `autocomplete` to it only if its CPU latency is acceptable for you; `msci-code-fast` is about
   3× faster.

## Done when

- [ ] vLLM installs and the training libraries still import
- [ ] the starting checkpoint is chosen from section 2
- [ ] the teacher projection fits your compute budget
- [ ] the smoke run finishes all steps and its model card renders
- [ ] the full run passes its gates (or the failures are understood) and is registered